In [1]:
import kagglehub

path = kagglehub.dataset_download("omkargurav/face-mask-dataset")

print("path to dataset file: ",path)

100%|██████████| 163M/163M [00:01<00:00, 160MB/s]

Extracting files...


path to dataset file:  /root/.cache/kagglehub/datasets/omkargurav/face-mask-dataset/versions/1


In [2]:
import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import keras
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Dense, Flatten

In [3]:
os.listdir(path + '/data')

['without_mask', 'with_mask']

In [4]:
len(os.listdir(path + '/data/with_mask'))

3725

In [5]:
len(os.listdir(path + '/data/without_mask'))

3828

In [6]:
train_datagen= ImageDataGenerator(
    rescale=1./255,
    rotation_range=40,
    horizontal_flip=True,
    zoom_range=0.2,
    validation_split=0.2
)

test_datagen= ImageDataGenerator(
    rescale=1./255
)


In [7]:
train_generator= train_datagen.flow_from_directory(
    directory= path+ '/data',
    target_size=(128,128),
    batch_size=32,
    class_mode='binary',
    subset='training'
)

Found 6043 images belonging to 2 classes.


In [8]:
test_generator = train_datagen.flow_from_directory(
    directory= path+'/data',
    target_size=(128,128),
    batch_size=32,
    class_mode='binary',
    subset='validation'
)

Found 1510 images belonging to 2 classes.


In [9]:
model= tf.keras.Sequential([
    Conv2D(filters=32,kernel_size=(3,3),activation='relu',input_shape=(128,128,3)),
    MaxPooling2D(pool_size=(2,2)),
    Conv2D(filters=32,kernel_size=(3,3),activation='relu'),
    MaxPooling2D(pool_size=(2,2)),
    Conv2D(filters=64,kernel_size=(3,3),activation='relu'),
    MaxPooling2D(pool_size=(2,2)),
    Flatten(),
    Dense(128,activation='relu'),
    Dense(1,activation='sigmoid')
])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [10]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 126, 126, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 63, 63, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 61, 61, 32)     │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 30, 30, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 28, 28, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 14, 14, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 12544)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │     1,605,760 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,634,529 (6.24 MB)

 Trainable params: 1,634,529 (6.24 MB)

 Non-trainable params: 0 (0.00 B)

In [11]:
model.compile(optimizer='adam',loss='binary_crossentropy',metrics=['accuracy'])

In [12]:
history=model.fit(train_generator,epochs=10,validation_data=test_generator)

Epoch 1/10
 84/189 ━━━━━━━━━━━━━━━━━━━━ 1:28 847ms/step - accuracy: 0.7036 - loss: 0.5570

/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


189/189 ━━━━━━━━━━━━━━━━━━━━ 178s 932ms/step - accuracy: 0.8309 - loss: 0.3791 - val_accuracy: 0.8689 - val_loss: 0.3079
Epoch 2/10
189/189 ━━━━━━━━━━━━━━━━━━━━ 176s 934ms/step - accuracy: 0.8762 - loss: 0.2967 - val_accuracy: 0.8967 - val_loss: 0.2554
Epoch 3/10
189/189 ━━━━━━━━━━━━━━━━━━━━ 174s 922ms/step - accuracy: 0.8981 - loss: 0.2477 - val_accuracy: 0.9139 - val_loss: 0.2122
Epoch 4/10
189/189 ━━━━━━━━━━━━━━━━━━━━ 174s 921ms/step - accuracy: 0.9010 - loss: 0.2447 - val_accuracy: 0.9179 - val_loss: 0.2232
Epoch 5/10
189/189 ━━━━━━━━━━━━━━━━━━━━ 173s 916ms/step - accuracy: 0.9110 - loss: 0.2217 - val_accuracy: 0.9377 - val_loss: 0.1807
Epoch 6/10
189/189 ━━━━━━━━━━━━━━━━━━━━ 172s 910ms/step - accuracy: 0.9242 - loss: 0.1898 - val_accuracy: 0.9371 - val_loss: 0.1795
Epoch 7/10
189/189 ━━━━━━━━━━━━━━━━━━━━ 173s 914ms/step - accuracy: 0.9249 - loss: 0.1880 - val_accuracy: 0.9338 - val_loss: 0.1813
Epoch 8/10
189/189 ━━━━━━━━━━━━━━━━━━━━ 210s 955ms/step - accuracy: 0.9326 - loss: 0.16